In [1]:
!pip install chromadb
!pip install llama_index
!pip install llama-index-vector-stores-chroma
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-openrouter
!pip install PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 1.7 MB/s eta 0:00:0

In [2]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.llms import ChatMessage
from llama_index.core.postprocessor import LongContextReorder
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import (
    StorageContext,
    Settings,
    VectorStoreIndex,
    SimpleDirectoryReader
    )

import PyPDF2
import torch
import re
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
def is_scanned_pdf(pdf_path, page_sample=10, threshold=50):
    """Проверяет PDF на содержание изображений вместо текста"""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)

        checked_pages = min(page_sample, len(reader.pages))
        if checked_pages == 0:
            return False

        empty_count = 0
        for i in range(checked_pages):
            text = reader.pages[i].extract_text()
            if len(text.strip()) < threshold:
                empty_count += 1

        return empty_count == checked_pages

def pdf2txt(pdf_path):
    """Извлечение текста из PDF с помощью PyPDF2"""

    if is_scanned_pdf(pdf_path):
        print(f"Внимание: {pdf_path} похож на сканированный документ! Текст не может быть извлечен.")
        return ""

    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n\n"
    return text

def clean_for_rag(text):
    cleaned_text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s,.!?;:—]+', '', text)  # Оставляем буквы и нужные знаки
    cleaned_text = cleaned_text.strip()  # Удаляем лишние пробелы в начале и конце
    cleaned_text = re.sub(r'\n\s*\n', '\n\n', cleaned_text)
    return cleaned_text

def process_document(file_path):
    if file_path.lower().endswith('.pdf'):
        text = pdf2txt(file_path)
        if not text:
            print(f"Файл {file_path} не был обработан (причина: сканированный документ)")
            return None
    else:
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            text = f.read()
    text = clean_for_rag(text)
    os.makedirs("data", exist_ok=True)
    filepath = os.path.join("data", "processed_text.txt")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Файл {file_path} успешно обработан")

process_document('11-0.txt')

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USER_PROMPT = """Твоя задача — отвечать **ТОЛЬКО** на основе предоставленного контекста.

**Строгие правила:**
1. Если ответа нет в контексте — говори: «Информация не найдена в предоставленных материалах».
2. Запрещено придумывать факты, даже если они кажутся правдоподобными.
3. Если контекст противоречив — укажи на это и процитируй разные точки зрения.

Формат ответа:
- Четко, кратко, без лишних слов, на **РУССКОМ ЯЗЫКЕ**.
- Если нужно перечислить пункты — используй маркированные списки.
- Код, формулы или специфичные термины выделяй `как код`."""


# MODEL_RAG = "deepseek/deepseek-chat-v3-0324:free" # лучше качество, меньше скорость
MODEL_RAG = 'mistralai/mistral-nemo:free' # чуть похуже, но быстрее

API_KEY_RAG = ''


class RAGSystem:
    def __init__(
            self,
            api_key=None, model_rag=None,
            collection_name="IQ-Group",
            embed_model_name="intfloat/multilingual-e5-small"
            ):


        self._initialize_chroma(collection_name)
        self._setup_embeddings(embed_model_name)
        self._load_documents()
        self.llm(api_key, model_rag)

    def llm(self,api_key: str,model_rag: str):
        """Настройка модели"""
        Settings.llm = OpenRouter(
            max_tokens=10000,
            context_window=30000,
            temperature=0,
            api_key=api_key,
            model=model_rag,
        )
        self._setup_query_engine()

    def _initialize_chroma(self, collection_name: str):
        """Инициализация ChromaDB"""
        client = chromadb.PersistentClient(path="./chroma_db")
        collection = client.get_or_create_collection(collection_name)

        # Настройка хранилища векторов
        vector_store = ChromaVectorStore(chroma_collection=collection)
        self.storage_context = StorageContext.from_defaults(vector_store=vector_store)

    def _setup_embeddings(self, embed_model_name: str):
        """Загрузка модели для эмбеддингов"""
        self.embed_model = HuggingFaceEmbedding(
            model_name=embed_model_name,
            device=DEVICE,
            model_kwargs={"torch_dtype": "auto"}
        )
        Settings.embed_model = self.embed_model

    def _load_documents(self):
        """Загрузка документов из директории"""
        pipeline = IngestionPipeline(
            transformations=[
                SentenceSplitter(
                    chunk_size=512,
                    chunk_overlap=20,
                    paragraph_separator=r'\n\s*\n',  # Учитываем абзацы
                    secondary_chunking_regex=r'(?<=[.!?])\s+',  # Разделитель предложений
                ),
                # TitleExtractor(),#только с OpenAI
                self.embed_model,
            ]
        )
        documents = SimpleDirectoryReader("./data").load_data()
        nodes = pipeline.run(documents=documents)
        self.index = VectorStoreIndex(
            nodes=nodes,
            storage_context=self.storage_context,
            embed_model=self.embed_model
        )

    def _setup_query_engine(self):
        """Настройка query engine"""
        reorder = LongContextReorder()
        self.query_engine = self.index.as_query_engine(
            similarity_top_k=5,
            node_postprocessors=[reorder],
            streaming=True,
        )

    def format_query(self, query: str) -> str:
        """Форматирование пользовательского запроса"""
        return f"{USER_PROMPT}\n\nВопрос: {query}"


    def ask(self, query: str):
        """Выполнение запроса к системе"""
        formatted_query = self.format_query(query)
        response = self.query_engine.query(formatted_query)
        for token in response.response_gen:
            print(token, end="", flush=True)




rag = RAGSystem(
        api_key=API_KEY_RAG,
        model_rag=MODEL_RAG
    )


# Стриминг ответа
rag.ask("О чем документ?")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Документ содержит отрывки из произведения "Алиса в Зазеркалье" Льюиса Кэрролла. В нем описывается приключение Алисы в мире, где все вокруг странно и нелогично. Алиса встречает различных персонажей, в том числе Короля, Королеву, Чeshire-кота, Мартовского зайца, Белого кролика и других. В тексте описываются их диалоги и действия, а также происходящие вокруг них события.

In [ ]:
rag.llm(API_KEY_RAG,"deepseek/deepseek-chat-v3-0324:free")

In [ ]:
print('Спросите меня что-нибудь о документе:\n')
while True:
    query=input('Введите вопрос: (0-завершить)\n')
    if query!='0':
        rag.ask(query)
    else:
        break

Спросите меня что-нибудь о документе:

Введите вопрос: (0-завершить)
О чем документ?
Документ представляет собой отрывки из истории о приключениях Алисы, где она взаимодействует с различными фантастическими персонажами, такими как:  
- Мышь, с которой у неё происходит неловкий разговор  
- Чеширский Кот, обсуждающий безумие и играющий в прятки  
- Король и Королева, ведущие абсурдный суд  
- Голубь, обвиняющий Алису в том, что она змея  

Основные темы:  
* Абсурдные диалоги и логические головоломки  
* Превращения и странные события  
* Конфликты с капризными персонажами  
* Игра в крокет с Королевой  

Стиль повествования — сюрреалистичная сказка с элементами нонсенса.Введите вопрос: (0-завершить)
0
